# [의료보험료 예측 딥러닝 모델링 심화 (PyTorch MLP Regression - Ver 2.0)]

## 프로젝트 배경 및 목표
본 프로젝트는 피보험자의 인적/신체적 조건(나이, 성별, BMI, 자녀 수, 흡연 여부 등)을 바탕으로 책정될 의료보험료(Charges)를 예측하는 딥러닝 회귀(Regression) 파이프라인 구축 실습입니다.

<br>

초기 베이스라인 모델(Ver 1.0)에서는 '흡연자이면서 비만'인 케이스를 추출한 파생 변수(`High_Risk`)를 힌트로 제공하여 R2 Score 0.83이라는 우수한 성능을 달성했습니다. 이번 **Ver 2.0 (End-Game)** 에서는 단순한 피처 추가를 넘어, 데이터의 비선형적 패턴을 모델에 명시적으로 학습시키고 이상치(Outlier)에 대한 모델의 방어력을 높여 **결정계수(R2) 0.9 이상 도달을 목표로 하는 최적화 기법**을 적용합니다.

<br>

---

<br>

## Ver 2.0 핵심 개선 사항 (Advanced Techniques)

**1. 교호작용 파생 변수 (Interaction Features) 도입**
* 변수들이 독립적으로 작용하지 않고 서로 결합하여 발생하는 시너지 효과를 모델에 직접 전달합니다.
* `Age_x_Smoker`: 비흡연자 대비 흡연자의 '연령 증가에 따른 가파른 보험료 상승 폭'을 반영합니다.
* `BMI_x_Smoker`: 흡연 여부에 따라 기하급수적으로 달라지는 'BMI의 가중치'를 반영합니다.

<br>

**2. Huber Loss (Smooth L1 Loss) 채택**
* 기존에 사용한 MSE(평균 제곱 오차)는 극단적인 고액 보험료(이상치)에 과도한 페널티를 부여하여, 대다수 일반 환자에 대한 전체 모델의 설명력을 떨어뜨리는 딜레마가 있었습니다.
* 이를 보완하기 위해 오차가 작을 때는 MSE처럼 둥글게, 오차가 클 때는 MAE처럼 직선으로 동작하는 **Huber Loss**를 도입하여 이상치에 대한 저항력(Robustness)을 대폭 향상시켰습니다.

<br>

**3. Learning Rate Scheduler (`ReduceLROnPlateau`) 도입**
* 고정된 보폭(학습률)으로 인한 오버슈팅을 방지합니다. 검증 손실(Validation Loss)이 더 이상 떨어지지 않고 정체되는 구간(Plateau)에서 학습률을 동적으로 감소(Decay)시켜, 최적점(Local Minima)에 세밀하게 안착하도록 유도합니다.
<br>

**4. Optuna 기반 하이퍼파라미터 자동 최적화 (AutoML)**
* 파생 변수 추가로 늘어난 입력 차원(Input Dimension)에 맞춰, Optuna를 활용해 최적의 은닉층 아키텍처(노드 수)와 초기 학습률(Learning Rate), 배치 사이즈를 자동으로 탐색하여 파이프라인에 적용합니다.



#1. interaction 파생 변수 추가 및 통합 전처리

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

data_path = '/content/drive/MyDrive/Colab Notebooks/PyTorch 실습 /insurance.csv'

df = pd.read_csv(data_path)

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from collections import defaultdict

# ---------------------------------------------------------
# [A] 인코더 클래스 정의
# ---------------------------------------------------------
class MultiLabelEncoder:
    def __init__(self):
        self.label_encoders = defaultdict(LabelEncoder)
    def fit(self, X, columns):
        for column in columns: self.label_encoders[column].fit(X[column])
        return self
    def transform(self, X, columns):
        X_transformed = X.copy()
        for column in columns: X_transformed[column] = self.label_encoders[column].transform(X[column])
        return X_transformed

class DataFrameOneHotEncoder:
    def __init__(self):
        self.encoder = OneHotEncoder(sparse_output=False)
    def fit(self, X, columns):
        self.columns = columns
        self.encoder.fit(X[columns])
        self.feature_names = self.encoder.get_feature_names_out(columns)
        return self
    def transform(self, X):
        X_transformed = X.drop(columns=self.columns).reset_index(drop=True)
        encoded_columns = pd.DataFrame(self.encoder.transform(X[self.columns]), columns=self.feature_names)
        return pd.concat([X_transformed, encoded_columns], axis=1)

# ---------------------------------------------------------
# [B] 데이터 로드 및 🚀 고급 파생 변수(Interaction) 추가
# ---------------------------------------------------------
# df = pd.read_csv(data_path) # 데이터 로드가 필요하다면 주석 해제

# 흡연 여부를 0과 1로 변환 (계산용)
smoker_binary = (df['smoker'] == 'yes').astype(int)

# 1. High Risk (흡연 + 비만)
df['high_risk'] = (smoker_binary & (df['bmi'] >= 30.0)).astype(int)
# 2. 나이 x 흡연 교호작용 (흡연자의 나이에 따른 가파른 보험료 상승분 반영)
df['age_x_smoker'] = df['age'] * smoker_binary
# 3. BMI x 흡연 교호작용
df['bmi_x_smoker'] = df['bmi'] * smoker_binary

print("✅ 파생 변수 3종 추가 완료!")

# ---------------------------------------------------------
# [C] Train / Test Split (순정 타겟) 및 인코딩, 스케일링
# ---------------------------------------------------------
x = df.copy().drop(columns=['charges'])
y = df.copy()['charges']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2024)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=2024)

# 인코딩 적용
label_cols = ['sex', 'smoker']
multilabelencoder = MultiLabelEncoder().fit(x_train, label_cols)
x_train = multilabelencoder.transform(x_train, label_cols)
x_test = multilabelencoder.transform(x_test, label_cols)
x_val = multilabelencoder.transform(x_val, label_cols)

oe_cols = ['region']
onehotencoder = DataFrameOneHotEncoder().fit(x_train, oe_cols)
x_train = onehotencoder.transform(x_train.copy())
x_test = onehotencoder.transform(x_test.copy())
x_val = onehotencoder.transform(x_val.copy())

# 스케일링 적용
scaler = StandardScaler()
numeric_cols = list(set(x_train.select_dtypes(exclude="object").columns))

scaler.fit(x_train[numeric_cols])
x_train[numeric_cols] = scaler.transform(x_train[numeric_cols].copy())
x_test[numeric_cols] = scaler.transform(x_test[numeric_cols].copy())
x_val[numeric_cols] = scaler.transform(x_val[numeric_cols].copy())

# Tensor 변환
x_train_tensor, y_train_tensor = torch.Tensor(x_train.values), torch.Tensor(y_train.values).unsqueeze(1)
x_val_tensor, y_val_tensor = torch.Tensor(x_val.values), torch.Tensor(y_val.values).unsqueeze(1)
x_test_tensor, y_test_tensor = torch.Tensor(x_test.values), torch.Tensor(y_test.values).unsqueeze(1)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

input_dim = x_train.shape[1]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ 전처리 완료! 현재 Input Dimension: {input_dim}")

✅ 파생 변수 3종 추가 완료!
✅ 전처리 완료! 현재 Input Dimension: 12


#2. Optuna 하이퍼파라미터 자동 탐색 (Huber Loss 손실 함수)

In [5]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.6 MB/s eta 0:00:00


In [6]:
import optuna

class OptunaV2Model(nn.Module):
    def __init__(self, trial, input_dim):
        super(OptunaV2Model, self).__init__()
        n_units_l1 = trial.suggest_int("n_units_l1", 32, 128)
        n_units_l2 = trial.suggest_int("n_units_l2", 16, 64)
        n_units_l3 = trial.suggest_int("n_units_l3", 8, 32)

        self.linear_1 = nn.Linear(input_dim, n_units_l1)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(n_units_l1, n_units_l2)
        self.linear_3 = nn.Linear(n_units_l2, n_units_l3)
        self.linear_4 = nn.Linear(n_units_l3, 1)

    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.relu(self.linear_3(x))
        return self.linear_4(x)

def objective_v2(trial):
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    epochs = 200

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = OptunaV2Model(trial, input_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 🚀 핵심: Huber Loss 적용 (이상치 페널티 완화)
    loss_fn = nn.HuberLoss().to(device)

    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    val_preds_list, val_trues_list = [], []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            preds = model(x_batch.to(device)).cpu().numpy()
            val_preds_list.extend(preds)
            val_trues_list.extend(y_batch.numpy())

    return mean_squared_error(val_trues_list, val_preds_list)

print("🚀 Ver 2.0 하이퍼파라미터 탐색 시작...")
study_v2 = optuna.create_study(direction="minimize")
study_v2.optimize(objective_v2, n_trials=30)

best_params = study_v2.best_params
print("\n🏆 최고 성능 하이퍼파라미터 조합:")
for key, value in best_params.items(): print(f"  {key}: {value}")

[I 2026-05-13 04:30:49,574] A new study created in memory with name: no-name-e6713378-5945-45be-8ae4-1c23db2d8adb


🚀 Ver 2.0 하이퍼파라미터 탐색 시작...


[I 2026-05-13 04:31:40,349] Trial 0 finished with value: 17774949.85008784 and parameters: {'lr': 0.07402293122774033, 'batch_size': 64, 'n_units_l1': 126, 'n_units_l2': 58, 'n_units_l3': 18}. Best is trial 0 with value: 17774949.85008784.
[I 2026-05-13 04:31:58,411] Trial 1 finished with value: 18624273.739649747 and parameters: {'lr': 0.026545675643849784, 'batch_size': 16, 'n_units_l1': 36, 'n_units_l2': 30, 'n_units_l3': 16}. Best is trial 0 with value: 17774949.85008784.
[I 2026-05-13 04:32:09,421] Trial 2 finished with value: 18645482.84319733 and parameters: {'lr': 0.0024551434373704666, 'batch_size': 32, 'n_units_l1': 79, 'n_units_l2': 60, 'n_units_l3': 31}. Best is trial 0 with value: 17774949.85008784.
[I 2026-05-13 04:32:14,804] Trial 3 finished with value: 19571076.253092997 and parameters: {'lr': 0.04574499756537018, 'batch_size': 64, 'n_units_l1': 52, 'n_units_l2': 55, 'n_units_l3': 14}. Best is trial 0 with value: 17774949.85008784.
[I 2026-05-13 04:32:33,129] Trial 4 fi


🏆 최고 성능 하이퍼파라미터 조합:
  lr: 0.07402293122774033
  batch_size: 64
  n_units_l1: 126
  n_units_l2: 58
  n_units_l3: 18


#3. 최종 학습 (스케쥴러 적용) 및 평가

In [7]:
# 1. 최적 파라미터 자동 반영
batch_size = best_params['batch_size']
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. 최적 아키텍처 선언
class FinalV2Model(nn.Module):
    def __init__(self, input_dim):
        super(FinalV2Model, self).__init__()
        self.linear_1 = nn.Linear(input_dim, best_params['n_units_l1'])
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(best_params['n_units_l1'], best_params['n_units_l2'])
        self.linear_3 = nn.Linear(best_params['n_units_l2'], best_params['n_units_l3'])
        self.linear_4 = nn.Linear(best_params['n_units_l3'], 1)

    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.relu(self.linear_3(x))
        return self.linear_4(x)

model = FinalV2Model(input_dim).to(device)

# 3. 옵티마이저, Huber Loss, 그리고 🚀 LR Scheduler 선언
optimizer = optim.Adam(params=model.parameters(), lr=best_params['lr'])
loss_fn = nn.HuberLoss().to(device)

# 스케쥴러: Validation Loss가 15번의 에포크 동안 개선이 없으면 학습률을 절반(0.5)으로 줄임
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

epochs = 500
print(f"\n🚀 최종 V2 모델 학습 시작 (Batch: {batch_size}, LR: {best_params['lr']:.5f})")

for epoch in range(epochs):
    # Train Phase
    model.train()
    train_cost = 0
    for train_inputs, train_labels in train_dataloader:
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        optimizer.zero_grad()
        train_preds = model(train_inputs)
        train_loss = loss_fn(train_preds, train_labels)
        train_loss.backward()
        optimizer.step()
        train_cost += train_loss.item()

    train_cost /= len(train_dataloader)

    # Validation Phase (스케쥴러가 사용할 성능 모니터링)
    model.eval()
    val_cost = 0
    with torch.no_grad():
        for val_inputs, val_labels in val_dataloader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
            val_loss = loss_fn(model(val_inputs), val_labels)
            val_cost += val_loss.item()
    val_cost /= len(val_dataloader)

    # 스케쥴러 업데이트 (검증 손실 기반)
    scheduler.step(val_cost)

    if (epoch + 1) % 100 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch: {epoch+1:3d}/{epochs} | Train Loss(Huber): {train_cost:.2f} | Val Loss: {val_cost:.2f} | LR: {current_lr:.6f}")

# ---------------------------------------------------------
# 🏆 최종 Evaluation
# ---------------------------------------------------------
model.eval()
with torch.no_grad():
    test_preds = model(x_test_tensor.to(device)).cpu().numpy()
    test_trues = y_test_tensor.numpy()

mse = mean_squared_error(test_trues, test_preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_trues, test_preds)
r2 = r2_score(test_trues, test_preds)

print("\n" + "="*50)
print(" 🚀 의료보험료 예측 V2 (End-Game) 최종 평가 결과")
print("="*50)
print(f" ▪ MSE  (평균 제곱 오차)  : {mse:,.2f}")
print(f" ▪ RMSE (평균 제곱근 오차): {rmse:,.2f}")
print(f" ▪ MAE  (평균 절대 오차)  : {mae:,.2f}")
print(f" ▪ R2 Score (결정계수)    : {r2:.4f}")
print("="*50)


🚀 최종 V2 모델 학습 시작 (Batch: 64, LR: 0.07402)
Epoch: 100/500 | Train Loss(Huber): 1371.03 | Val Loss: 1493.71 | LR: 0.009253
Epoch: 200/500 | Train Loss(Huber): 1296.58 | Val Loss: 1452.23 | LR: 0.001157
Epoch: 300/500 | Train Loss(Huber): 1257.85 | Val Loss: 1448.02 | LR: 0.000145
Epoch: 400/500 | Train Loss(Huber): 1345.69 | Val Loss: 1447.83 | LR: 0.000005
Epoch: 500/500 | Train Loss(Huber): 1233.61 | Val Loss: 1447.83 | LR: 0.000000

 🚀 의료보험료 예측 V2 (End-Game) 최종 평가 결과
 ▪ MSE  (평균 제곱 오차)  : 24,627,090.00
 ▪ RMSE (평균 제곱근 오차): 4,962.57
 ▪ MAE  (평균 절대 오차)  : 1,486.96
 ▪ R2 Score (결정계수)    : 0.8353


### ver 2.0의 핵심 향상 포인트
1. 평균 예측 오차(MAE)의 유의미한 감소

- 지표 변화: Ver 0 (3,190달러) ➡️ Ver 1 (2,794달러) ➡️ Ver 2 (1,486달러)

- 결과 해석: 기존 모델들이 건당 평균 2,800~3,100달러 수준의 오차를 보인 반면, Ver 2.0은 이를 약 1,486달러로 대폭 축소했습니다. 이는 극단적인 비용이 발생하는 소수의 케이스를 제외한, 대다수 일반 가입자 그룹에 대한 예측 신뢰도가 실무 적용이 가능할 수준으로 향상되었음을 의미합니다.

<br>

2. Huber Loss 도입을 통한 이상치 강건성(Robustness) 확보

- 지표 비교: Ver 1과 Ver 2의 R2 Score(0.836 vs 0.835)와 RMSE(4,947 vs 4,962)는 유사한 수준을 방어했습니다.

- 결과 해석: MSE 대신 Huber Loss를 적용한 전략이 효과적으로 작동했습니다. 극단적으로 높은 보험료를 가진 소수의 이상치(Outlier)에 모델의 가중치가 과도하게 쏠리는 과적합(Overfitting)을 방지함으로써, 전체 데이터의 설명력(R2)을 훼손하지 않으면서도 전반적인 예측 오차(MAE)를 안정화할 수 있었습니다.

<br>

3. 교호작용(Interaction) 변수 추가를 통한 비선형적 패턴 학습

- 결과 해석: Age_x_Smoker와 BMI_x_Smoker 변수를 추가하여 변수 간의 시너지 효과를 모델에 반영했습니다. 이를 통해 단순한 피처의 합을 넘어, "흡연 여부에 따라 연령과 BMI가 보험료 상승에 미치는 영향(가중치)이 상이하게 나타난다"는 데이터의 비선형적 특성을 모델이 학습하게 되었고, 이것이 오차율 개선의 주요 요인으로 작용했습니다.